# HDSoC ROOT data overview

Explore current HD/HT banks. In periodic mode, one MIDAS event contains many triggers; each waveform is identified by channel and trigger timestamp.


In [ ]:
# Edit this path, then run the notebook from the top.
# Absolute paths and ~ are supported; relative paths start at the app directory.
DATA_FILE = ".artifacts/bank_format_verification/run121479/output.root"


In [ ]:
from pathlib import Path
import sys, os
# Works when Jupyter starts in the repository or a notebook subdirectory.
APP_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                if (p / 'notebooks/notebook_support.py').is_file())
if str(APP_ROOT / 'notebooks') not in sys.path:
    sys.path.insert(0, str(APP_ROOT / 'notebooks'))
from notebook_support import (open_data, present, hit_waveform, hit_baseline,
    hit_amplitude, hit_peak, waveform_entries as find_waveform_entries,
    sampic_unit, integer, waveform_packets, sampic_event_bytes)
import ROOT
r = ROOT
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
root_file, tree, DATA_FILE = open_data('HDSoC', DATA_FILE)
f = file = root_file
plt.rcParams.update({'figure.figsize': (12, 5), 'axes.grid': True, 'grid.alpha': 0.25})


In [ ]:
print('Branches:', [b.GetName() for b in tree.GetListOfBranches()])
root_file.ls()
if tree.GetEntries():
    tree.GetEntry(0)
    event = tree.nalu_event
    print('First event:', event.header.event_index, 'packets:', event.header.num_packets,
          'windows per waveform:', integer(event.header.num_windows), 'waveforms:', len(event.waveforms.waveforms))
    print('Collection cycle:', tree.nalu_time.time.collection_cycle_index)


In [ ]:
# HT can be repeated on multiple events emitted by one collection cycle.
# Deduplicate by cycle index + steady-clock timestamp before cycle histograms.
cycles = {}
for entry in range(tree.GetEntries()):
    tree.GetEntry(entry)
    t = tree.nalu_time.time
    cycles[(int(t.collection_cycle_index),int(t.collection_cycle_timestamp_ns))] = (
        float(t.total_time),int(t.data_processed),float(t.data_rate))
fig, axes = plt.subplots(1,3,figsize=(14,4))
for i,label in enumerate(('Cycle processing [s]','Processed bytes / cycle','Collector processing rate [MiB/s]')):
    axes[i].hist([x[i] for x in cycles.values()],bins=40); axes[i].set(xlabel=label,ylabel='Unique cycles')
plt.tight_layout(); plt.show()
print('Unique collection cycles:', len(cycles))


In [ ]:
fig, ax = plt.subplots()
for entry in range(min(3,tree.GetEntries())):
    tree.GetEntry(entry)
    for wf in list(tree.nalu_event.waveforms.waveforms)[:3]:
        ax.plot(np.array(wf.trace,dtype=np.uint16,copy=True),alpha=.7,
                label=f'Entry {entry}, ch {integer(wf.channel_num)}, trigger {wf.trigger_time}')
ax.set(xlabel='Sample index',ylabel='ADC counts')
if ax.lines: ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
